# Stage2 / 01 — Inspect branch configs + warm-start inputs

Loads each Stage2 YAML, prints the **delta vs the raw 1c YAML**,
and verifies that every `WARM_START_CKPT` / `DISTILL_CKPT` referenced
actually exists on Drive. Saves a summary JSON so notebooks 02-05
can trust the config surface they load.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
!pip install -q yacs tqdm

In [ ]:
from lib.config import cfg
from pathlib import Path
from copy import deepcopy

CONFIGS = [
    'stage2/configs/detrlane_vehicle1c.yaml',
    'stage2/configs/detrlane_vehicle1c_warmstart.yaml',
    'stage2/configs/detrlane_vehicle3c_warmstart.yaml',
    'stage2/configs/detrlane_vehicle3c_distill.yaml',
    'stage2/configs/detrlane_vehicle3c_distill_gradnorm.yaml',
]

report = {}
for rel in CONFIGS:
    cfg.defrost()
    cfg.merge_from_file(os.path.join(REPO_ROOT, rel))
    ckpt = getattr(cfg.STAGE2, 'WARM_START_CKPT', '') or ''
    dckpt = getattr(cfg.STAGE2, 'DISTILL_CKPT', '') or ''
    report[rel] = {
        'NC': cfg.MODEL.NC,
        'CLASS_PROTOCOL': cfg.DATASET.CLASS_PROTOCOL,
        'WARM_START': bool(cfg.STAGE2.WARM_START),
        'WARM_START_CKPT_exists': os.path.exists(ckpt) if ckpt else False,
        'DISTILL': bool(cfg.STAGE2.DISTILL),
        'DISTILL_CKPT_exists': os.path.exists(dckpt) if dckpt else False,
        'GRAD_BALANCE': cfg.STAGE2.GRAD_BALANCE,
        'END_EPOCH': cfg.TRAIN.END_EPOCH,
        'SGDR': cfg.TRAIN.SGDR,
    }
    print(rel, '=>', report[rel])
    cfg.freeze()
    cfg = deepcopy(cfg)          # reset for next iter; yacs freeze is sticky

out = os.path.join('/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane',
                   'stage2', 'metrics', 'branch_config_report.json')
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'w') as f:
    json.dump(report, f, indent=2)
print('Saved ->', out)